# Lab 09｜令行禁止：怎样用隔离与审批守住授权边界？

这是专栏《自进化 Agent 实战》09 讲《令行禁止：用隔离与审批守住授权边界》的配套练习。练习使用构造的 hotfix 材料，目标是掌握处理真实场景的能力。

## 题面

一次小改动，计划写明只动一个适配层目录和它的测试目录，其余代码只读、不联网。执行中 Agent 发现一行就能改完共享连接池的更省事方案，那是别的业务也在用的公共代码。计划里其实已经写着"别碰共享代码"，可它还是试了。你会怎样设计执行环境，让"别碰"不是一句写给模型看的话，而是真的落不下去？又由谁来决定这份权限配置生不生效？

## 参考答案

先分清两件事：计划告诉 Agent 应该怎么做，执行环境保证哪些修改真能落地。Agent 执行中会不断拿到新信息、临场重新判断，这是它的价值；不能跟着一起变的是权限。光在计划或方法里写一句"不许改共享代码"不够，模型可能判断错，它发出的命令还可能顺手拉起别的子进程。真正不许发生的修改，必须在执行环境这一层拦住。

我会用"默认拒绝、显式放行"的边界：工作目录本来就在可写范围；这次确实要动的测试目录，再额外开一个可写根；剩下所有位置默认只读，联网一并关死。这样边界是闭合的，不用成天担心漏掉哪个"不许碰"的路径。

至于配置怎么来、谁批准：Agent 可以照着计划把要求翻译成沙箱配置，但不能自己批准自己的权限。顺序应当是 Agent 先形成计划、我确认；Agent 再生成配置、我再核对它和计划能不能逐条对上——计划说改两处，可写范围就恰好盖住这两处，不多出第三个位置。应用前还能做个自检：在适配层建临时文件应成功，到共享目录建应失败，边界才算真的生效。

审批处理的是另一类情况：平时关着、看现场临时放行一次的例外，放出去的应是一次性例外而不是长期通行证，任务一结束权限收回。要是发现自己几乎每次都在批同一件事，那该回头改的是计划和默认边界，而不是一次次点同意把边界磨穿。

## 准备

使用 Python 3.12；实际模型运行需要 `GLM_API_KEY`，未设置时弹出一次隐藏输入，依赖自动补装。实例名、三处调用与 30 秒、5 秒均为构造值。
本讲直接使用底层 `CodexClient`，把完整的 `sandboxPolicy` 传给 `turn_start()`；后面另建带 `approval_handler` 的客户端演示一次性审批。准备脚本只管理模型连接与材料；`lab.watch()` 保护等待，`lab.record_result()` 保存结果。离线只回放材料，不启动模型。


In [ ]:
import sys
from pathlib import Path
examples = next(p for base in (Path.cwd(), *Path.cwd().parents)
                for p in (base, base / "examples") if (p / "notebook09_setup.py").is_file())
sys.path.insert(0, str(examples))
from notebook09_setup import prepare
story = prepare()

## 阶段一｜最省事的方案，为什么不能让它自己改？

从三层构造代码开始：只读形成计划，按题设已确认的范围生成配置，独立核对后再执行 Hotfix。
这里起作用的是 Sandbox 可写根与默认拒绝：cwd 适配层可写，测试额外放行，共享 lib 写入被拒后回到三个调用点。
观察允许目录、共享文件哈希和拒绝记录；offline 真实回放文件变化，原生隔离效果只在 live 中核验。

### 先把执行边界写成具体参数

`cwd` 是适配层目录，`writableRoots` 只额外加入对应测试目录。这里的范围由题设预先确认，模型生成的草案必须经过独立核对后才能传给执行端。


In [ ]:
import subprocess, re, json
from copy import deepcopy
from openai_codex.client import CodexClient
from openai_codex.api import Thread, TurnHandle
from notebook_support import read_json_answer, command_receipts, exact_command
lab, work = story.lab, story.work
readonly = {"cwd": str(work), "approvalPolicy": "never", "sandboxPolicy": {"type": "readOnly"}}
expected = {"cwd": str(story.outputs), "approvalPolicy": "never", "sandboxPolicy": {
    "type": "workspaceWrite", "writableRoots": [str(story.tests)], "networkAccess": False,
    "excludeTmpdirEnvVar": True, "excludeSlashTmp": True}}
if lab:
    client = lab.manage_client(CodexClient(config=lab.config))
    client.start()
    client.initialize()
    story._policy_proc = getattr(client, "_proc", None)
def record_policy(permission):
    story.submitted_policies.append(deepcopy(permission))
    story.save("submitted-policies.json", story.submitted_policies)


In [ ]:
plan = draft = None
if lab:
    response = client.thread_start({"cwd": str(work), "sandbox": "read-only", "approvalPolicy": "never"})
    planner_id = response.thread.id
    response = client.turn_start(planner_id, story.planning_task, readonly)
    record_policy(readonly)
    turn = TurnHandle(client, planner_id, response.turn.id)
    with lab.watch(turn):
        plan = turn.run()
    lab.record_result(turn, plan, show_result=False)
    response = client.turn_start(planner_id, story.proposal_task, readonly)
    record_policy(readonly)
    turn = TurnHandle(client, planner_id, response.turn.id)
    with lab.watch(turn):
        draft = turn.run()
    lab.record_result(turn, draft, show_result=False)
    m = re.search(r"\{.*\}", draft.final_response or "", re.S)
    try:
        proposal = read_json_answer(m.group(0) if m else "", ["cwd", "approvalPolicy", "sandboxPolicy"])
    except AssertionError:
        proposal = story.config_from_materials()
else:
    proposal = story.config_from_materials()
permission = story.review_scope(proposal)
if permission != expected:
    raise ValueError("独立核对后的配置仍与题设不符，停止执行。")


In [ ]:
start = len(lab.requests) if lab else 0
result = None
if lab:
    response = client.thread_start({"cwd": permission["cwd"], "sandbox": "read-only", "approvalPolicy": "never"})
    scoped_id = response.thread.id
    task = (f"先原样执行一次 `{story.scoped_command}`；共享写入被拒后不重试、不扩权。"
        f"回到 {story.adapter}，三个 acquire 调用显式传入 wait_seconds=5，补充 {story.regression} 并运行回归。"
        "保留共享默认值30秒；只改这两个目录，不联网。测试只核对参数，不模拟真实等待。")
    response = client.turn_start(scoped_id, task, permission)
    record_policy(permission)
    turn = TurnHandle(client, scoped_id, response.turn.id)
    with lab.watch(turn):
        result = turn.run()
    lab.record_result(turn, result, show_result=False)
story.show_execution(result, start, permission)


## 阶段二｜计划里已经说“不要改”，为什么还不够？

这里起作用的是执行环境拦截：计划、Skill 和项目规则管模型怎么想，Sandbox 管哪些修改真能落地。
在只读配置下尝试写入，再故意把共享目录加进配置，观察拒绝与原权限未变；运行参数回归核对 Hotfix。
临场方案可以变化，权限不能漂移；offline 的教学函数不能证明任意 Shell 或子进程已经受操作系统隔离。

In [ ]:
start = len(lab.requests) if lab else 0
readonly_result = None
if lab:
    response = client.turn_start(planner_id,
        f"原样执行一次 `{story.readonly_command}`，拒绝后停止，不重试、不申请扩权。", readonly)
    record_policy(readonly)
    turn = TurnHandle(client, planner_id, response.turn.id)
    with lab.watch(turn):
        readonly_result = turn.run()
    lab.record_result(turn, readonly_result, show_result=False)
story.record_readonly(readonly_result, start)


In [ ]:
regression = subprocess.run([sys.executable, "-B", str(story.regression)], cwd=work,
                            capture_output=True, text=True)
print(regression.stdout + regression.stderr)
story.show_plan_boundary(plan, regression)

## 阶段三｜怎样把计划真正变成权限？

这里起作用的是配置生成与人审边界：回看 Agent 的配置草案，逐项对照计划，再应用同组权限做临时文件自检。
适配层建文件应成功，共享目录应失败；另一个独立任务演示审批一次性例外，主 Hotfix 的 never 范围不变。
人决定配置是否生效；litellm 代理层只负责模型协议适配，offline 不启动它，也不把回放当成真实审批请求。

In [ ]:
permission = story.review_scope(proposal)
print(json.dumps(permission, ensure_ascii=False, indent=2))
start = len(lab.requests) if lab else 0
selfcheck = None
if lab:
    response = client.thread_start({"cwd": permission["cwd"], "sandbox": "read-only", "approvalPolicy": "never"})
    checked_id = response.thread.id
    response = client.turn_start(checked_id,
        f"只原样执行一次 `{story.selfcheck_command}`，共享写入被拒后立即停止，不重试、不扩权。", permission)
    record_policy(permission)
    turn = TurnHandle(client, checked_id, response.turn.id)
    with lab.watch(turn):
        selfcheck = turn.run()
    lab.record_result(turn, selfcheck, show_result=False)
story.record_selfcheck(selfcheck, start)


### 审批回调怎样只消费一次批准

这是独立的构造报告任务。主 Hotfix 的 `approvalPolicy="never"` 保持原值。回调只接受完整命令、工作目录和脚本哈希都匹配的请求；批准一次后立即清零。这里用预先确认的教学批准演示消费过程，生产中该批准应来自人或独立审批系统。


In [ ]:
import hashlib, shlex
def review_command(method, params):
    params = params or {}
    raw = params.get("command", "")
    try:
        argv = shlex.split(raw) if isinstance(raw, str) else raw
        if (isinstance(argv, list) and len(argv) == 3
                and argv[0] in ("/bin/bash", "/bin/zsh", "/bin/sh", "bash", "zsh", "sh")
                and argv[1] in ("-c", "-lc")):
            argv = shlex.split(argv[2])
        exact = argv == ["python3", "write_report.py"]
    except (ValueError, TypeError):
        exact = False
    matched = (method == "item/commandExecution/requestApproval" and exact
        and params.get("cwd") == str(story.outputs)
        and story.script.is_file() and not story.script.is_symlink()
        and hashlib.sha256(story.script.read_bytes()).hexdigest() == story.script_hash)
    permitted = story.approved and matched
    if permitted:
        story.approved = False
    decision = "accept" if permitted else "decline"
    story.approvals.append({"source": "native-client" if lab else "offline-callback-replay",
        "method": method, "command": story.clean(raw), "cwd": story.clean(params.get("cwd")),
        "matched": matched, "decision": decision})
    story.save("approvals.json", story.approvals)
    return {"decision": decision}


### 先拒绝：没有批准就不写报告

把上面的函数传给 `CodexClient(approval_handler=...)`，由 SDK 收到原生审批请求后调用它。第一个独立 Thread 没有一次性批准，回调应返回 `decline`。


In [ ]:
task = "请只运行 python3 write_report.py，不读取或改写脚本；审批拒绝立即停止，成功后报告状态。"
approval_config = {"cwd": str(story.outputs), "sandbox": "workspace-write",
    "approvalPolicy": "untrusted", "approvalsReviewer": "user",
    "developerInstructions": "只执行用户给出的那条命令；若审批拒绝立即停止，不换工具或命令。"}
if lab:
    approval_client = lab.manage_client(CodexClient(config=lab.config, approval_handler=review_command))
    approval_client.start()
    approval_client.initialize()
    story._approval_proc = getattr(approval_client, "_proc", None)
    response = approval_client.thread_start(approval_config)
    denied = Thread(approval_client, response.thread.id)
    turn = denied.turn(task)
    with lab.watch(turn):
        result = turn.run()
    lab.record_result(turn, result, show_result=False)
    absent = not story.report.exists()


### 批准一次，再检查批准是否已经消费

第二个独立 Thread 使用同一个回调；只有这一次可以匹配并消费批准。两次执行都有真实 SDK 请求。最后再次直接调用回调，检查已消费的批准不能重复使用；这一步是回调自检，不是第三次模型执行。


In [ ]:
if lab:
    story.approved = True
    response = approval_client.thread_start(approval_config)
    accepted = Thread(approval_client, response.thread.id)
    start = len(lab.requests)
    try:
        turn = accepted.turn(task)
        with lab.watch(turn):
            result = turn.run()
        lab.record_result(turn, result, show_result=False)
        consumed = not story.approved
    finally:
        story.approved = False
    receipts = command_receipts(lab.requests[start:])
    repeated = review_command("item/commandExecution/requestApproval",
        {"command": "python3 write_report.py", "cwd": str(story.outputs)})["decision"]
    story.save("approval-result.json", {"source": "native-client", "absent_after_denial": absent,
        "exit_code": 0 if any(r["exit_code"] == 0 and exact_command(r["command"], "python3 write_report.py") for r in receipts) else None,
        "repeat_decision": repeated, "consumed": consumed, "thread_ids": [denied.id, accepted.id], "receipts": receipts})
else:
    story.replay_approval(review_command)
story.finish()
